## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv('/content/drive/MyDrive/HW_Pandas/cookie_cats.csv')

In [4]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# параметри
p1 = 0.19
p2 = 0.20
alpha = 0.05
power = 0.8

# effect size
effect_size = proportion_effectsize(p1, p2)

# розмір вибірки
analysis = NormalIndPower()
sample_size = analysis.solve_power(effect_size=effect_size,
                                   power=power,
                                   alpha=alpha,
                                   ratio=1)

print("Sample size per group:", round(sample_size))
print("Total sample size:", round(sample_size * 2))

Sample size per group: 24638
Total sample size: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [5]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [6]:
df.groupby('version')['retention_7'].mean()

,retention_7
version,
gate_30,0.190201
gate_40,0.182000


Середнє значення retention_7 для версії gate_30 становить 0.1902, а для gate_40 — 0.1820. Тому версія gate_30 має вище утримання користувачів.
Гіпотези:
H0: retention_7 у версії gate_40 менше або дорівнює retention_7 у версії gate_30.
Ha: retention_7 у версії gate_40 більше, ніж у версії gate_30.
Далі необхідно перевірити цю гіпотезу статистично.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [7]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control = df[df['version'] == 'gate_30']['retention_7']
treatment = df[df['version'] == 'gate_40']['retention_7']

count = [control.sum(), treatment.sum()]

nobs = [len(control), len(treatment)]

z_stat, p_value = proportions_ztest(count, nobs)

print("z statistic:", z_stat)
print("p-value:", p_value)

z statistic: 3.164358912748191
p-value: 0.001554249975614329


In [8]:
ci_control = proportion_confint(count[0], nobs[0], alpha=0.05, method='normal')
ci_treatment = proportion_confint(count[1], nobs[1], alpha=0.05, method='normal')

print("Довірчий інтервал 95% для групи control:", ci_control)
print("Довірчий інтервал 95% для групи treatment:", ci_treatment)

Довірчий інтервал 95% для групи control: (0.18656311652199903, 0.19383956804175934)
Довірчий інтервал 95% для групи treatment: (0.17845430073314686, 0.18554578720019968)


1) Оскільки p-value < 0.05, ми відхиляємо нульову гіпотезу. Це означає, що між версіями гри існує статистично значуща різниця в утриманні користувачів на 7 день. 2) Довірчі інтервали для двох груп не перетинаються. Це додатково підтверджує, що різниця між групами є статистично значущою.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [9]:
import pandas as pd
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df['version'], df['retention_7'])
print(contingency_table)

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279


In [10]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi2 statistic:", chi2)
print("p-value:", p_value)

Chi2 statistic: 9.959086799559167
p-value: 0.0016005742679058301


In [11]:
alpha = 0.05

if p_value < alpha:
    print("Відхиляємо H0")
else:
    print("Не відхиляємо H0")

Відхиляємо H0


За результатами χ² тесту отримане p-value < 0.05, тому ми відхиляємо нульову гіпотезу. Це означає, що існує статистично значуща залежність між версією гри та утриманням користувачів на 7 день.

Отже, зміна розташування воріт у грі вплинула на поведінку користувачів.